### Import libraries

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import patsy
from scipy.special import expit
from scipy.stats import chi2
from statsmodels.discrete.truncated_model import TruncatedLFNegativeBinomialP

from pyprojroot import here 

### Load datasets

In [2]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"
PROCESSED_DATA_DIR = here() / "data" / "processed"
RESULTS_DATA_DIR = here() / "results"

In [3]:
df = pd.read_csv(INTERIM_DATA_DIR/"crime_with_population.csv", encoding="latin-1")

In [4]:
df

,CVEGEO,year,state,municipality,crime_count,population,log_pop
0,1001,2015,Aguascalientes,Aguascalientes,863,873000.0,13.679691
1,1002,2015,Aguascalientes,Asientos,8,48514.0,10.789608
2,1003,2015,Aguascalientes,Calvillo,17,56193.0,10.936547
3,1004,2015,Aguascalientes,CosÃ­o,14,16021.0,9.681656
4,1005,2015,Aguascalientes,JesÃºs MarÃ­a,24,114759.5,11.650594
...,...,...,...,...,...,...,...
27253,32054,2025,Zacatecas,Villa Hidalgo,2,19924.0,9.899680
27254,32055,2025,Zacatecas,Villanueva,36,32639.5,10.393278
27255,32056,2025,Zacatecas,Zacatecas,213,155322.5,11.953259
27256,32057,2025,Zacatecas,Trancoso,8,22215.5,10.008546


In [5]:
df["log_population"] = np.log(df["population"])

## Poisson study

In [6]:
poisson = smf.glm(
    formula="crime_count ~ C(year)",
    data=df,
    family=sm.families.Poisson(),
    offset=df["log_population"]
).fit()

print(poisson.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:            crime_count   No. Observations:                27029
Model:                            GLM   Df Residuals:                    27018
Model Family:                 Poisson   Df Model:                           10
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -7.4670e+05
Date:                Mon, 21 Sep 2026   Deviance:                   1.4376e+06
Time:                        21:02:50   Pearson chi2:                 2.41e+06
No. Iterations:                     6   Pseudo R-squ. (CS):             0.8883
Covariance Type:            nonrobust                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -7.7813      0.004  -17

In [7]:
pearson_dispersion = (
    poisson.pearson_chi2 /
    poisson.df_resid
)

print("Pearson chi-square:", poisson.pearson_chi2)
print("Residual df:", poisson.df_resid)
print("Dispersion:", pearson_dispersion)

Pearson chi-square: 2412302.8155017225
Residual df: 27018
Dispersion: 89.28502537203799


In [8]:
deviance_dispersion = (
    poisson.deviance /
    poisson.df_resid
)

print("Deviance/df:", deviance_dispersion)

Deviance/df: 53.21072857705262


In [9]:
df["poisson_fitted"] = poisson.fittedvalues
df["poisson_pearson_resid"] = poisson.resid_pearson
df["poisson_deviance_resid"] = poisson.resid_deviance

In [10]:
print(
    df[
        [
            "poisson_fitted",
            "poisson_pearson_resid",
            "poisson_deviance_resid"
        ]
    ].describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

       poisson_fitted  poisson_pearson_resid  poisson_deviance_resid
count    27029.000000           27029.000000            27029.000000
mean        36.353213              -1.773701               -2.785327
std        108.747140               9.279324                6.740379
min          0.034282             -28.715811              -35.593522
1%           0.244378             -11.598029              -14.296150
5%           0.611291              -6.822714               -8.623316
25%          2.996615              -3.589450               -4.676132
50%          9.175464              -2.191389               -2.878326
75%         24.401924              -1.170875               -1.550056
95%        141.710978               2.072899                1.885018
99%        570.806893              22.341364               17.978300
max       1941.440380             519.524894              271.233643


In [11]:
print("Pearson |r| > 2:",
      (df["poisson_pearson_resid"].abs() > 2).sum())

print("Pearson |r| > 3:",
      (df["poisson_pearson_resid"].abs() > 3).sum())

print("Pearson |r| > 5:",
      (df["poisson_pearson_resid"].abs() > 5).sum())

print("Maximum Pearson residual:",
      df["poisson_pearson_resid"].max())

print("Minimum Pearson residual:",
      df["poisson_pearson_resid"].min())

Pearson |r| > 2: 16084
Pearson |r| > 3: 10372
Pearson |r| > 5: 4130
Maximum Pearson residual: 519.5248943795715
Minimum Pearson residual: -28.71581113333667


#### Poisson excess zeros analysis

In [12]:
observed_zeros = np.sum(df["crime_count"] == 0)

predicted_means = poisson.predict()

expected_zeros = np.sum(np.exp(-predicted_means))

print(f"Observed Zeros in Data: {observed_zeros}")
print(f"Expected Zeros from Poisson GLM: {round(expected_zeros, 2)}")

zero_ratio = observed_zeros / expected_zeros
print(f"Zero-Inflation Ratio: {round(zero_ratio, 2)}")

Observed Zeros in Data: 12524
Expected Zeros from Poisson GLM: 2251.72
Zero-Inflation Ratio: 5.56


## Zero Inflated Poisson

In [13]:
df_phase2 = (
    df[
        df["crime_count"].notna()
        & df["population"].notna()
        & (df["population"] > 0)
    ]
    .copy()
)

df_phase2["log_population"] = np.log(
    df_phase2["population"]
)

print(df_phase2.shape)

(27029, 11)


In [14]:
y_train, X_train = patsy.dmatrices(
    "crime_count ~ C(year)",
    data=df_phase2,
    return_type="dataframe"
)

y_train = y_train.values.ravel()

In [15]:
zi_poisson = sm.ZeroInflatedPoisson(
    endog=y_train, 
    exog=X_train, 
    exog_infl=X_train, 
    offset=df_phase2["log_population"].values,
    inflation='logit'
).fit(maxiter=100, method='bfgs') 

print(zi_poisson.summary())

Optimization terminated successfully.
         Current function value: 25.594029
         Iterations: 49
         Function evaluations: 52
         Gradient evaluations: 52
                     ZeroInflatedPoisson Regression Results                    
Dep. Variable:                       y   No. Observations:                27029
Model:             ZeroInflatedPoisson   Df Residuals:                    27018
Method:                            MLE   Df Model:                           10
Date:                 Mon, 21 Sep 2026   Pseudo R-squ.:                 0.03627
Time:                         21:02:55   Log-Likelihood:            -6.9178e+05
converged:                        True   LL-Null:                   -7.1782e+05
Covariance Type:             nonrobust   LLR p-value:                     0.000
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
inf

## Negative Binomial study

In [16]:
nb = smf.negativebinomial(
    formula="crime_count ~ C(year)",
    data=df,
    offset=df["log_population"]
).fit(
    method="bfgs",
    maxiter=1000,
    disp=True
)

print(nb.summary())

Optimization terminated successfully.
         Current function value: 2.346962
         Iterations: 33
         Function evaluations: 34
         Gradient evaluations: 34
                     NegativeBinomial Regression Results                      
Dep. Variable:            crime_count   No. Observations:                27029
Model:               NegativeBinomial   Df Residuals:                    27018
Method:                           MLE   Df Model:                           10
Date:                Mon, 21 Sep 2026   Pseudo R-squ.:                0.006063
Time:                        21:02:58   Log-Likelihood:                -63436.
converged:                       True   LL-Null:                       -63823.
Covariance Type:            nonrobust   LLR p-value:                8.428e-160
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -8.7773  

In [17]:
print(nb.mle_retvals)

{'fopt': np.float64(2.3469621639352938), 'gopt': array([ 1.50842373e-06,  9.55860153e-06, -4.70707871e-06, -3.04213813e-06,
       -3.23493386e-07, -5.69623671e-07, -2.32299290e-07, -5.53439295e-07,
       -8.14451308e-07, -6.05281890e-07, -3.78966075e-06, -4.51357041e-07]), 'Hinv': array([[ 2.17186240e+01, -2.71656332e+01, -1.84811847e+01,
        -1.94360474e+01, -2.12289534e+01, -2.10167748e+01,
        -2.13924374e+01, -2.10717455e+01, -2.08324242e+01,
        -2.10052244e+01, -1.81159262e+01, -7.20827188e-02],
       [-2.71656332e+01,  5.25108984e+01,  2.22888119e+01,
         3.13511625e+01,  2.67488722e+01,  2.53952780e+01,
         3.07271481e+01,  2.67799022e+01,  2.51339755e+01,
         2.57704712e+01,  2.70199870e+01, -4.51445492e-02],
       [-1.84811847e+01,  2.22888119e+01,  2.18231014e+01,
         2.28723200e+01,  1.98585087e+01,  1.94297062e+01,
         2.16015843e+01,  2.04733113e+01,  1.99568718e+01,
         1.97864617e+01,  1.41518910e+01, -5.25417682e-02],
     

In [18]:
print(nb.params)

Intercept         -8.777277
C(year)[T.2016]   -0.051822
C(year)[T.2017]    0.178437
C(year)[T.2018]    0.358943
C(year)[T.2019]    0.563090
C(year)[T.2020]    0.564162
C(year)[T.2021]    0.572802
C(year)[T.2022]    0.632508
C(year)[T.2023]    0.649427
C(year)[T.2024]    0.666019
C(year)[T.2025]    0.902080
alpha              2.229149
dtype: float64


In [19]:
alpha = nb.params["alpha"]
phi = 1 / alpha

print(f"alpha = {alpha:.6f}")
print(f"phi = {phi:.6f}")

alpha = 2.229149
phi = 0.448602


Negative Binomial exccess zeros study

In [20]:
nb_predicted_means = nb.predict()

nb_expected_zeros = np.sum(np.exp(-nb_predicted_means))

print(f"Observed Zeros in Data: {observed_zeros}")
print(f"Expected Zeros from Negative Binomial SMF: {round(nb_expected_zeros, 2)}")

nb_zero_ratio = observed_zeros / nb_expected_zeros
print(f"Zero-Inflation Ratio: {round(nb_zero_ratio, 2)}")

Observed Zeros in Data: 12524
Expected Zeros from Negative Binomial SMF: 5376.93
Zero-Inflation Ratio: 2.33


Results justify testing a Zero-Inflated Negative Binomial Model

## Zero Inflated Negative Binomial

In [21]:
X_infl_const = np.ones((len(df_phase2), 1))

In [22]:
zinb = sm.ZeroInflatedNegativeBinomialP(
    endog=y_train,
    exog=X_train,
    exog_infl=X_infl_const,
    offset=df_phase2["log_population"].values,
    inflation='logit',
    p=2
).fit(maxiter=200, method='bfgs')

Optimization terminated successfully.
         Current function value: 2.346962
         Iterations: 65
         Function evaluations: 66
         Gradient evaluations: 66


In [23]:
zinb.summary()

C:\Users\franc\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
C:\Users\franc\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


<class 'statsmodels.iolib.summary.Summary'>
"""
                     ZeroInflatedNegativeBinomialP Regression Results                    
=========================================================================================
Dep. Variable:                                 y   No. Observations:                27029
Model:             ZeroInflatedNegativeBinomialP   Df Residuals:                    27018
Method:                                      MLE   Df Model:                           10
Date:                           Mon, 21 Sep 2026   Pseudo R-squ.:                0.006063
Time:                                   21:03:21   Log-Likelihood:                -63436.
converged:                                  True   LL-Null:                       -63823.
Covariance Type:                       nonrobust   LLR p-value:                8.428e-160
===================================================================================
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
inflate_const     -20.5441    515.716     -0.040      0.968   -1031.330     990.241
Intercept          -8.7775      0.034   -260.120      0.000      -8.844      -8.711
C(year)[T.2016]    -0.0518      0.048     -1.084      0.278      -0.145       0.042
C(year)[T.2017]     0.1790      0.047      3.778      0.000       0.086       0.272
C(year)[T.2018]     0.3591      0.047      7.630      0.000       0.267       0.451
C(year)[T.2019]     0.5633      0.047     12.012      0.000       0.471       0.655
C(year)[T.2020]     0.5644      0.047     12.044      0.000       0.473       0.656
C(year)[T.2021]     0.5731      0.047     12.231      0.000       0.481       0.665
C(year)[T.2022]     0.6329      0.047     13.537      0.000       0.541       0.725
C(year)[T.2023]     0.6498      0.047     13.906      0.000       0.558       0.741
C(year)[T.2024]     0.6662      0.047     14.228      0.000       0.574       0.758
C(year)[T.2025]     0.9024      0.047     19.349      0.000       0.811       0.994
alpha               2.2292      0.026     85.795      0.000       2.178       2.280
===================================================================================
"""

In [24]:
print("P(structural zero) =", expit(-20.5441))

P(structural zero) = 1.1962191036123268e-09


#### Zero-Inflated Negative Binomial excess zero test

In [25]:
observed_zeros = np.sum(y_train == 0)

predicted_zero_probabilities = zinb.predict(which="prob-zero")

expected_zeros = np.sum(predicted_zero_probabilities)

print(f"Observed Zeros in Data: {observed_zeros}")
print(f"Expected Zeros from ZINB Model: {round(expected_zeros, 2)}")

zero_ratio = observed_zeros / expected_zeros
print(f"Zero-Inflation Ratio: {round(zero_ratio, 2)}")

Observed Zeros in Data: 12360
Expected Zeros from ZINB Model: 11527.09
Zero-Inflation Ratio: 1.07


The Zero-Inflated model successfully solved the zero excess issue.

## Hurdle Negative Binomial

In [26]:
y_bin = (y_train > 0).astype(int)

In [27]:
hurdle_binom = sm.GLM(
    y_bin, X_train,
    family=sm.families.Binomial(),
    offset=df_phase2["log_population"].values
).fit()

In [28]:
# print(hurdle_binom.summary())

In [29]:
mask_pos = y_train > 0

In [30]:
trunc_nb = TruncatedLFNegativeBinomialP(
    endog=y_train[mask_pos],
    exog=X_train[mask_pos],
    offset=df_phase2["log_population"].values[mask_pos],
    truncation=0,
    p=2
).fit(maxiter=100, method='bfgs')

Optimization terminated successfully.
         Current function value: 3.426275
         Iterations: 35
         Function evaluations: 36
         Gradient evaluations: 36


In [31]:
print(trunc_nb.summary())

                    TruncatedLFNegativeBinomialP Regression Results                     
Dep. Variable:                                y   No. Observations:                14669
Model:             TruncatedLFNegativeBinomialP   Df Residuals:                    14657
Method:                                     MLE   Df Model:                           10
Date:                          Mon, 21 Sep 2026   Pseudo R-squ.:                0.003799
Time:                                  21:03:32   Log-Likelihood:                -50260.
converged:                                 True   LL-Null:                       -50452.
Covariance Type:                      nonrobust   LLR p-value:                 3.275e-76
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -8.7031      0.047   -185.780      0.000      -8.795      -8.611
C(year)[T.2016]    -0.0546      0.06

In [32]:
llf_hurdle = hurdle_binom.llf + trunc_nb.llf
k_hurdle = len(hurdle_binom.params) + len(trunc_nb.params)
n_hurdle = len(y_train)

In [33]:
aic_hurdle = -2 * llf_hurdle + 2 * k_hurdle
bic_hurdle = -2 * llf_hurdle + k_hurdle * np.log(n_hurdle)

## Comparison

In [32]:
def vuong_test(llf_i, llf_j):
    m = llf_i - llf_j
    n = len(m)
    stat = np.sqrt(n) * m.mean() / m.std(ddof=1)
    return stat

In [33]:
comparison = pd.DataFrame({
    "Model": ["Poisson", "Negative Binomial", "ZI Poisson", "Zero-Inflated Negative Binomial", "Hurdle Negaitve Binomial"],
    "LogLik": [poisson.llf, nb.llf, zi_poisson.llf, zinb.llf, llf_hurdle],
    "AIC": [poisson.aic, nb.aic, zi_poisson.aic, zinb.aic, aic_hurdle],
    "BIC": [poisson.bic_llf, nb.bic, zi_poisson.bic, zinb.bic, bic_hurdle]
})

print(comparison.round(2))

                             Model     LogLik         AIC         BIC
0                          Poisson -746699.95  1493421.90  1493512.15
1                Negative Binomial  -63436.04   126896.08   126994.54
2                       ZI Poisson -691781.01  1383606.02  1383786.53
3  Zero-Inflated Negative Binomial  -63436.04   126898.08   127004.74
4         Hurdle Negaitve Binomial  -62274.76   124595.51   124784.22


In [34]:
lr = 2 * (nb.llf - poisson.llf)

print(f"LR statistic = {lr:.3f}")

LR statistic = 1366527.816


In [36]:
# %%
mu_hurdle_bin = hurdle_binom.predict()  # fitted P(y_bin=1 | x)

llf_obs_hurdle = np.zeros(n_hurdle)
llf_obs_hurdle[:] = hurdle_binom.family.loglike_obs(y_bin, mu_hurdle_bin)
llf_obs_hurdle[mask_pos] += trunc_nb.model.loglikeobs(trunc_nb.params)

llf_nb_obs = nb.model.loglikeobs(nb.params)
llf_zinb_obs = zinb.model.loglikeobs(zinb.params)

v_stat_hurdle_nb = vuong_test(llf_obs_hurdle, llf_nb_obs)
v_stat_hurdle_zinb = vuong_test(llf_obs_hurdle, llf_zinb_obs)

print(f"Vuong statistic (Hurdle NB vs NB) = {v_stat_hurdle_nb:.3f}")
print(f"Vuong statistic (Hurdle NB vs ZINB) = {v_stat_hurdle_zinb:.3f}")

Vuong statistic (Hurdle NB vs NB) = 37.644
Vuong statistic (Hurdle NB vs ZINB) = 37.646


C:\Users\franc\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\discrete\discrete_model.py:3925: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  alpha = params[-1]
C:\Users\franc\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\discrete\discrete_model.py:4349: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  alpha = params[-1]
C:\Users\franc\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\discrete\discrete_model.py:3386: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labe

## Expected zeros

In [ ]:
Z_obs = (df_phase2["crime_count"] == 0).sum()
N = len(df_phase2)

print("N =", N)
print("Z_obs =", Z_obs)
print("Observed zero proportion =", Z_obs / N)

In [ ]:
mu_nb = nb.predict()

print(mu_nb[:10])
print("Number of predictions:", len(mu_nb))

In [ ]:
rng = np.random.default_rng(12345)

alpha = nb.params["alpha"]
r = 1 / alpha

mu_nb = np.asarray(nb.predict())

p = r / (r + mu_nb)

B = 5000

Z_rep = np.empty(B)

for b in range(B):

    y_rep = rng.negative_binomial(
        n=r,
        p=p
    )

    Z_rep[b] = np.sum(y_rep == 0)

In [ ]:
print("Observed zeros:", Z_obs)

print(
    "Mean replicated zeros:",
    Z_rep.mean()
)

print(
    "SD replicated zeros:",
    Z_rep.std()
)

print(
    "95% predictive interval:",
    np.quantile(Z_rep, [0.025, 0.975])
)

print(
    "99% predictive interval:",
    np.quantile(Z_rep, [0.005, 0.995])
)

In [ ]:
p_upper = np.mean(Z_rep >= Z_obs)
p_lower = np.mean(Z_rep <= Z_obs)

print("P(Z_rep >= Z_obs):", p_upper)
print("P(Z_rep <= Z_obs):", p_lower)

In [ ]:
plt.figure(figsize=(9, 5))

plt.hist(
    Z_rep,
    bins=40,
    alpha=0.75
)

plt.axvline(
    Z_obs,
    linestyle="--",
    linewidth=2,
    label=f"Observed = {Z_obs}"
)

plt.xlabel("Number of zero observations")
plt.ylabel("Frequency")
# plt.title(
#     "Predictive check of zero frequency\n"
#     "Negative Binomial model"
# )

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
B = len(Z_rep)

p_upper_corrected = (
    1 + np.sum(Z_rep >= Z_obs)
) / (B + 1)

print(
    "Corrected predictive p-value:",
    p_upper_corrected
)